# Download the RQ4 artefacts

Run this **in the same session** in which you ran the experiment.

It finds `rq4_results.json`, `rq4_manifest.json` and `raw_runs/`, prints the tool
versions recorded in the manifest so you can confirm it is the right run, packages
everything into `rq4_artifacts.zip`, and downloads it.

If it reports ARTEFACTS NOT FOUND, the runtime was restarted and `/content/` was
wiped — re-run the experiment notebook, then run this cell without restarting.


In [ ]:
# ===== locate and download the RQ4 artefacts =====
import os, glob, shutil, json

CANDIDATES = ["/content/rq4", "/content/rq4_out", "./rq4_out",
              "/content/drive/MyDrive/rq4", "/content/rq4_verify", "/content/rq4_expanded"]

found = None
for c in CANDIDATES:
    if os.path.isdir(c) and os.path.exists(os.path.join(c, "rq4_results.json")):
        found = c; break
if not found:
    hits = glob.glob("/content/**/rq4_results.json", recursive=True)
    if hits: found = os.path.dirname(hits[0])

if not found:
    print("=" * 66)
    print("ARTEFACTS NOT FOUND")
    print("=" * 66)
    print("The Colab session was almost certainly restarted — everything under")
    print("/content/ is deleted when the runtime disconnects or is reset.")
    print()
    print("What to do:")
    print("  1. Run the experiment notebook again (Runtime -> Run all).")
    print("  2. Come straight back to THIS cell and run it, without restarting.")
    print()
    print("Directories currently under /content:")
    for d in sorted(os.listdir("/content")):
        print("   ", d)
else:
    print("found artefacts in:", found)
    for f in ["rq4_results.json", "rq4_manifest.json"]:
        p = os.path.join(found, f)
        print(f"  {f:22s} {'OK' if os.path.exists(p) else 'MISSING'}")
    raw = os.path.join(found, "raw_runs")
    n_raw = len(glob.glob(os.path.join(raw, "*.json"))) if os.path.isdir(raw) else 0
    print(f"  raw_runs/              {n_raw} files")

    # quick integrity summary so you can confirm it is the right run
    try:
        man = json.load(open(os.path.join(found, "rq4_manifest.json")))
        print("\n  versions in manifest:")
        for k, v in (man.get("tool_versions") or {}).items():
            print(f"    {k:11s} {v}")
        print("  started:", man.get("started_utc"), "| runs:", man.get("total_tool_runs"))
    except Exception as e:
        print("  [!] could not read manifest:", e)

    out = "/content/rq4_artifacts"
    shutil.rmtree(out, ignore_errors=True)
    shutil.copytree(found, out)
    zip_path = shutil.make_archive("/content/rq4_artifacts", "zip", out)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"\npackaged: {zip_path} ({size_mb:.1f} MB)")

    try:
        from google.colab import files
        files.download(zip_path)
        print("download started — if the browser blocked it, use the Files pane on the left")
    except Exception as e:
        print("auto-download unavailable:", e)
        print("Open the Files pane (folder icon, left sidebar) and download rq4_artifacts.zip")
